In [2]:
from pathlib import Path

import cobra
import pandas as pd
cobra.Configuration().solver = "glpk"

pool_factors = [0.50, 0.75, 1.00, 1.25, 1.50]


def run_protein_pool_sensitivity(model, sources, base_media, calibrated_pool,
    carbon_bound=1000.0):

    output_dict = {}
    output_index = []

    # Get readable carbon-source names
    for source in sources:
        try:
            reaction = model.reactions.get_by_id(source)
            carbon_source_name = reaction.name.replace(' exchange', '')
            output_dict[carbon_source_name] = []

        except KeyError:
            print(f"{source} not found in model.")

    # Run each protein-pool multiplier
    for factor in pool_factors:
        pool_value = calibrated_pool * factor
        model.reactions.DM_usage.bounds = (-pool_value, 0.0)

        model_id = f"pool_{factor:.2f}x"
        output_index.append(model_id)

        print(
            f"\nRunning {model_id}: "
            f"DM_usage = {-pool_value}"
        )

        for source in sources:
            try:
                reaction = model.reactions.get_by_id(source)

            except KeyError:
                print(
                    f"Not testing {source}: "
                    f"exchange reaction not found."
                )
                continue

            carbon_source_name = reaction.name.replace(' exchange', '')

            media = base_media.copy()
            media.update({source: carbon_bound})

            with model:
                model.medium = media
                solution = model.optimize()

                if solution.status == "optimal":
                    value = solution.objective_value

                else:
                    value = float("nan")

                    print(
                        f"{model_id} failed on "
                        f"{source}: "
                        f"{solution.status}"
                    )

            output_dict[carbon_source_name].append(value)

    df = pd.DataFrame(output_dict, index=output_index).transpose()
    return df

In [5]:
"""
carved PAO1 - protein pool sensitivity
"""

# --------------------------------------------------------------
# SETTINGS
# --------------------------------------------------------------

base_media = {
    'EX_cd2_e': 1000.0,
    'EX_cobalt2_e': 1000.0,
    'EX_cu2_e': 1000.0,
    'EX_co2_e': 1000.0,
    # 'EX_fe2_e': 1000.0,
    'EX_fe3_e': 1000.0,
    'EX_h_e': 1000.0,
    'EX_h2o_e': 1000.0,
    'EX_k_e': 1000.0,
    'EX_mg2_e': 1000.0,
    'EX_mn2_e': 1000.0,
    # 'EX_n2_e': 1000.0,
    'EX_na1_e': 1000.0,
    'EX_nh4_e': 1000.0,
    'EX_o2_e': 1000.0,
    'EX_pi_e': 1000.0,
    'EX_so4_e': 1000.0,
    'EX_zn2_e': 1000.0,
    'EX_ca2_e': 1000.0,
    'EX_cl_e': 1000.0,
}

sources = [
    'EX_4abut_e',
    'EX_ac_e',
    'EX_cit_e',
    'EX_ala__D_e',
    'EX_fru_e',
    'EX_glc__D_e',
    'EX_mnl_e',
    'EX_man_e',
    'EX_glcn_e',
    'EX_glyc_e',
    'EX_ala__L_e',
    'EX_asp__L_e',
    'EX_his__L_e',
    'EX_lac__L_e',
    'EX_leu__L_e',
    'EX_orn_e',
    'EX_phe__L_e',
    'EX_pro__L_e',
    'EX_ser__L_e',
    'EX_akg_e',
    'EX_ptrc_e',
    'EX_succ_e',
    'EX_tre_e',
    'EX_asn__L_e',
    # 'EX_maln_e',
    'EX_cellb_e',
    'EX_h2o_e',
    'EX_glu__L_e',
    'EX_thr__L_e',
    # 'EX_btd__RR_e',
    'EX_crn_e',
]


ORIGINAL_EMMAI = Path(
    "../../ecGEMs_Paper/Submission_files/Supplementary file 1/PAO1/ecPAO1.xml"
)

OUTPUT_FILE = Path(
    "test_PAO1/predictions/protein_pool_sensitivity.csv"
)


# --------------------------------------------------------------
# PROTEIN POOL SETTINGS
# --------------------------------------------------------------

calibrated_pool = 1.35

# --------------------------------------------------------------
# OUTPUT STRUCTURE
# --------------------------------------------------------------

output_dict = {}
output_index = []

model = cobra.io.read_sbml_model(ORIGINAL_EMMAI)

for source in sources:

    try:
        reaction = model.reactions.get_by_id(source)
        carbon_source_name = reaction.name.replace(' exchange', '')
        output_dict[carbon_source_name] = []

    except KeyError:
        print(f"{source} not found in model.")

# --------------------------------------------------------------
# RUN PROTEIN-POOL SENSITIVITY
# --------------------------------------------------------------

for factor in pool_factors:
    pool_value = calibrated_pool * factor
    model.reactions.DM_usage.bounds = (-pool_value, 0.0)
    model_id = f"pool_{factor:.2f}x"
    output_index.append(model_id)

    print(
        f"\nRunning {model_id}: "
        f"DM_usage = {-pool_value}"
    )

    for source in sources:
        try:
            reaction = model.reactions.get_by_id(source)
            carbon_source_name = (reaction.name.replace(' exchange', ''))
            media = base_media.copy()
            media.update({source: 1000.0})

            with model:
                model.medium = media
                solution = model.optimize()
                output_dict[carbon_source_name].append(solution.objective_value)

        except KeyError:
            print(
                f"Not testing {source}: "
                f"exchange reaction not found."
            )

# --------------------------------------------------------------
# SAVE
# --------------------------------------------------------------

df = pd.DataFrame(output_dict, index=output_index)
df = df.transpose()
df.to_csv(OUTPUT_FILE)

print(
    f"\nSaved predictions to:\n"
    f"{OUTPUT_FILE}"
)

df


Running pool_0.50x: DM_usage = -0.675

Running pool_0.75x: DM_usage = -1.0125000000000002

Running pool_1.00x: DM_usage = -1.35


KeyboardInterrupt: 

In [3]:
"""
iML1515 and iJO1366 - protein pool sensitivity
"""


# --------------------------------------------------------------
# MEDIA AND CARBON SOURCES
# --------------------------------------------------------------

star_media = {
    'EX_ca2_e': 1000.0,
    'EX_cbl1_e': 0.01,
    'EX_cl_e': 1000.0,
    'EX_co2_e': 1000.0,
    'EX_cobalt2_e': 1000.0,
    'EX_cu2_e': 1000.0,
    'EX_fe2_e': 1000.0,
    'EX_fe3_e': 1000.0,
    'EX_h_e': 1000.0,
    'EX_h2o_e': 1000.0,
    'EX_k_e': 1000.0,
    'EX_mg2_e': 1000.0,
    'EX_mn2_e': 1000.0,
    'EX_mobd_e': 1000.0,
    'EX_nh4_e': 1000.0,
    'EX_ni2_e': 1000.0,
    'EX_o2_e': 1000.0,
    'EX_pi_e': 1000.0,
    'EX_so4_e': 1000.0,
    'EX_zn2_e': 1000.0,
}

sources = [
    'ac', 'glyc', 'akg', 'ala__L', 'pyr', 'fru', 'gsn',
    'acgam', 'fum', 'rib__D', 'lac__L', 'glcn', 'sbt__D',
    'gam', 'mal__L', 'succ', 'malt', 'g6p', 'man', 'tre',
    'xyl__D', 'gal', 'glc__D', 'mnl'
]


# --------------------------------------------------------------
# PATHS
# --------------------------------------------------------------

IJO_ORIGINAL_EMMAI = Path(
    "../../ecGEMs_Paper/Submission_files/Supplementary file 1/iJO1366/eciJO1366.xml"
)

IML_ORIGINAL_EMMAI = Path(
    "../../ecGEMs_Paper/Submission_files/Supplementary file 1/iML1515/eciML1515.xml"
)

OUTPUT_FILE = Path(
    "e_coli/result_comparison/"
    "protein_pool_sensitivity.csv"
)


# --------------------------------------------------------------
# PROTEIN POOL SETTINGS
# --------------------------------------------------------------

ijo_calibrated_pool = 0.68
iml_calibrated_pool = 0.99

# --------------------------------------------------------------
# LOAD MODELS
# --------------------------------------------------------------

ec_ijo_model = cobra.io.read_sbml_model(IJO_ORIGINAL_EMMAI)
ec_iml_model = cobra.io.read_sbml_model(IML_ORIGINAL_EMMAI)

# --------------------------------------------------------------
# OUTPUT STRUCTURE
# --------------------------------------------------------------

output_dict = {}
output_index = []

# Use iJO1366 to recover readable carbon-source names.

for source in sources:
    carbon_reaction = f'EX_{source}_e'
    reaction = ec_ijo_model.reactions.get_by_id(carbon_reaction)
    carbon_source_name = reaction.name.replace(' exchange', '')
    output_dict[carbon_source_name] = []

# --------------------------------------------------------------
# RUN ONE MODEL
# --------------------------------------------------------------

def run_model(model, model_id):
    output_index.append(model_id)
    for source in sources:
        carbon_reaction = f'EX_{source}_e'

        try:
            reaction = model.reactions.get_by_id(carbon_reaction)

        except KeyError:
            print(
                f"Not testing {source}: "
                f"exchange reaction not found "
                f"in {model_id}."
            )
            continue

        media = star_media.copy()
        media.update({carbon_reaction: 1000.0})

        with model:
            model.medium = media
            solution = model.optimize()
            if solution.status == "optimal":
                value = solution.objective_value

            else:
                value = float("nan")
                print(
                    f"{model_id} failed on "
                    f"{source}: "
                    f"{solution.status}"
                )

        carbon_source_name = (reaction.name.replace(' exchange', ''))
        output_dict[carbon_source_name].append(value)

# --------------------------------------------------------------
# iJO1366 PROTEIN POOL SENSITIVITY
# --------------------------------------------------------------

for factor in pool_factors:
    pool_value = (ijo_calibrated_pool * factor)
    ec_ijo_model.reactions.DM_usage.bounds = (-pool_value, 0.0)

    model_id = (f"iJO1366_pool_{factor:.2f}x")

    print(
        f"\nRunning {model_id}: "
        f"DM_usage = {-pool_value}"
    )

    run_model(ec_ijo_model, model_id)

# --------------------------------------------------------------
# iML1515 PROTEIN POOL SENSITIVITY
# --------------------------------------------------------------

for factor in pool_factors:
    pool_value = (iml_calibrated_pool * factor)
    ec_iml_model.reactions.DM_usage.bounds = (-pool_value, 0.0)

    model_id = (f"iML1515_pool_{factor:.2f}x")

    print(
        f"\nRunning {model_id}: "
        f"DM_usage = {-pool_value}"
    )

    run_model(ec_iml_model, model_id)

# --------------------------------------------------------------
# SAVE
# --------------------------------------------------------------

df = pd.DataFrame(output_dict, index=output_index)
df = df.transpose()
df.to_csv(OUTPUT_FILE)

print(
    f"\nSaved predictions to:\n"
    f"{OUTPUT_FILE}"
)


Running iJO1366_pool_0.50x: DM_usage = -0.34

Running iJO1366_pool_0.75x: DM_usage = -0.51

Running iJO1366_pool_1.00x: DM_usage = -0.68

Running iJO1366_pool_1.25x: DM_usage = -0.8500000000000001

Running iJO1366_pool_1.50x: DM_usage = -1.02

Running iML1515_pool_0.50x: DM_usage = -0.495

Running iML1515_pool_0.75x: DM_usage = -0.7424999999999999

Running iML1515_pool_1.00x: DM_usage = -0.99

Running iML1515_pool_1.25x: DM_usage = -1.2375

Running iML1515_pool_1.50x: DM_usage = -1.4849999999999999

Saved predictions to:
e_coli\result_comparison\protein_pool_sensitivity.csv


In [5]:
"""
iMO1056 - protein pool sensitivity
"""

ORIGINAL_MODEL = Path(
    "../../ecGEMs_Paper/Submission_files/"
    "Supplementary file 1/iMO1056/eciMO1056.xml"
)

OUTPUT_FILE = Path(
    "p_a01/predictions/"
    "iMO1056_protein_pool_sensitivity.csv"
)


base_media = {
    'EX_cd2_e': 1000.0,
    'EX_co2_e': 1000.0,
    'EX_cobalt2_e': 1000.0,
    'EX_cu2_e': 1000.0,
    'EX_fe2_e': 1000.0,
    'EX_fe3_e': 1000.0,
    'EX_h_e': 1000.0,
    'EX_h2o_e': 1000.0,
    'EX_k_e': 1000.0,
    'EX_mg2_e': 1000.0,
    'EX_mn2_e': 1000.0,
    'EX_n2_e': 1000.0,
    'EX_na1_e': 1000.0,
    'EX_nh4_e': 1000.0,
    'EX_o2_e': 1000.0,
    'EX_pi_e': 1000.0,
    'EX_so4_e': 1000.0,
    'EX_zn2_e': 1000.0,
}


sources = [
    'EX_4abut_e',
    'EX_ac_e',
    'EX_cit_e',
    'EX_ala__D_e',
    'EX_fru_e',
    'EX_glc_e',
    'EX_mnl_e',
    'EX_man_e',
    'EX_glcn__D_e',
    'EX_glyc_e',
    'EX_ala__L_e',
    'EX_asp__L_e',
    'EX_his__L_e',
    'EX_lac__L_e',
    'EX_leu__L_e',
    'EX_orn__L_e',
    'EX_phe__L_e',
    'EX_pro__L_e',
    'EX_ser__L_e',
    'EX_akg_e',
    'EX_ptrc_e',
    'EX_succ_e',
    'EX_tre_e',
    'EX_asn__L_e',
    'EX_maln_e',
    'EX_cellb_e',
    'EX_h2o_e',
    'EX_glu__L_e',
    'EX_thr__L_e',
    'EX_btd__RR_e',
    'EX_crn_e',
]


original_model = cobra.io.read_sbml_model(
    ORIGINAL_MODEL
)


df_imo = run_protein_pool_sensitivity(
    model=original_model,
    sources=sources,
    base_media=base_media,
    calibrated_pool=1.65,
)


df_imo.to_csv(
    OUTPUT_FILE
)

df_imo



Running pool_0.50x: DM_usage = -0.825

Running pool_0.75x: DM_usage = -1.2374999999999998

Running pool_1.00x: DM_usage = -1.65

Running pool_1.25x: DM_usage = -2.0625

Running pool_1.50x: DM_usage = -2.4749999999999996


,pool_0.50x,pool_0.75x,pool_1.00x,pool_1.25x,pool_1.50x
4-Aminobutanoate,2.998047e-01,4.497071e-01,5.996095e-01,7.495118e-01,8.994142e-01
Acetate,2.841353e-01,4.262029e-01,5.682705e-01,7.103382e-01,8.524058e-01
Citrate,5.005864e-01,7.508796e-01,1.001173e+00,1.251466e+00,1.501759e+00
D-Alanine,4.026168e-01,6.039252e-01,8.052336e-01,1.006542e+00,1.207850e+00
D-Fructose,5.940446e-01,8.910669e-01,1.188089e+00,1.485111e+00,1.782134e+00
D-Glucose,5.238685e-01,7.858027e-01,1.047737e+00,1.309671e+00,1.571605e+00
D-Mannitol,4.534688e-01,6.802033e-01,9.069377e-01,1.133672e+00,1.360407e+00
D-Mannose,-2.888550e-15,4.503596e-16,5.420392e-16,9.527311e-15,8.477278e-15
D-Gluconate,7.418165e-01,1.112725e+00,1.483633e+00,1.854541e+00,2.225450e+00
Glycerol,5.539459e-01,8.309189e-01,1.107892e+00,1.384865e+00,1.661838e+00


In [8]:
"""
ecBALT - protein pool sensitivity
"""


ORIGINAL_MODEL = Path(
    "../../ecGEMs_Paper/Submission_files/Supplementary file 1/BALT/ec_BALT_gapfilled.xml"
)

BIOLOG_FILE = Path(
    "../bsf_biolog/HILSE-31/HILSE-31_growth_growth_rates.xlsx"
)

OUTPUT_FILE = Path(
    "protein_pool_sensitivity/ecBALT.csv"
)


base_media = {
    'EX_cd2_e': 1000.0,
    'EX_cobalt2_e': 1000.0,
    'EX_cu2_e': 1000.0,
    'EX_fe2_e': 1000.0,
    'EX_fe3_e': 1000.0,
    'EX_h_e': 1000.0,
    'EX_h2o_e': 1000.0,
    'EX_k_e': 1000.0,
    'EX_mg2_e': 1000.0,
    'EX_mn2_e': 1000.0,
    # 'EX_n2_e': 1000.0,
    'EX_na1_e': 1000.0,
    'EX_nh4_e': 1000.0,
    'EX_o2_e': 1000.0,
    'EX_pi_e': 1000.0,
    'EX_so4_e': 1000.0,
    'EX_zn2_e': 1000.0,
    'EX_ca2_e': 1000.0,
    'EX_cl_e': 1000.0,
}


# Load Biolog data
biolog_31 = pd.read_excel(BIOLOG_FILE, index_col=22)

# Convert Biolog exchange IDs to BiGG format
sources = []
reaction_map = {}

for reaction in biolog_31.index:
    if isinstance(reaction, str):
        reaction_bigg = reaction.replace("(e)", "_e")
        reaction_map[reaction_bigg] = reaction

sources = list(reaction_map.keys())

# Load EMMAi model
balt_model = cobra.io.read_sbml_model(ORIGINAL_MODEL)

# Run protein-pool sensitivity
df_balt = run_protein_pool_sensitivity(
    model=balt_model,
    sources=sources,
    base_media=base_media,
    calibrated_pool=0.105,
)

# --------------------------------------------------------------
# ADD EXPERIMENTAL DATA
# --------------------------------------------------------------

carbon_sources = []
mean_max_rates = []

for source in sources:
    reaction_original = reaction_map[source]
    carbon_sources.append(biolog_31.at[reaction_original, "carbon_source"])
    mean_max_rates.append(biolog_31.at[reaction_original, "mean_max_rate"])

df_balt.insert(0, "carbon_source", carbon_sources)
df_balt.insert(1, "mean_max_rate", mean_max_rates)

# Save
df_balt.to_csv(OUTPUT_FILE)
df_balt


Running pool_0.50x: DM_usage = -0.0525

Running pool_0.75x: DM_usage = -0.07875

Running pool_1.00x: DM_usage = -0.105

Running pool_1.25x: DM_usage = -0.13125

Running pool_1.50x: DM_usage = -0.1575


,carbon_source,mean_max_rate,pool_0.50x,pool_0.75x,pool_1.00x,pool_1.25x,pool_1.50x
EX_arab__L_e,L-Arabinose,0.064203,0.033081,0.049622,0.066162,0.082703,0.099243
EX_acgam_e,N-Acetyl-D-Glucosamine,0.058524,0.019546,0.029319,0.039092,0.048865,0.058637
EX_succ_e,Succinic Acid,0.052045,0.008977,0.013466,0.017955,0.022443,0.026932
EX_gal_e,D-Galactose,0.053492,0.012841,0.019261,0.025682,0.032102,0.038523
EX_asp__L_e,L-Aspartic Acid,0.079607,0.014697,0.022046,0.029395,0.036743,0.044092
EX_pro__L_e,L-Proline,0.050514,0.019495,0.029243,0.038990,0.048738,0.058485
EX_ala__D_e,D-Alanine,0.048038,0.011319,0.016978,0.022637,0.028296,0.033956
EX_tre_e,D-Trehalose,0.068458,0.041214,0.061821,0.082428,0.103035,0.123642
EX_man_e,D-Mannose,0.053902,0.046855,0.070282,0.093709,0.117137,0.140564
EX_glyc_e,Glycerol,0.055154,0.038871,0.058307,0.077743,0.097179,0.116614


In [10]:
"""
PRET - protein pool sensitivity
"""

ORIGINAL_MODEL = Path(
    "../../ecGEMs_Paper/Submission_files/Supplementary file 1/PRET/ec_PRET_gapfilled.xml"
)

BIOLOG_FILE = Path(
    "../bsf_biolog/HILSE-52/HILSE-52_growth_growth_rates.xlsx"
)

OUTPUT_FILE = Path(
    "protein_pool_sensitivity/ecPRET.csv"
)


base_media = {
    'EX_cd2_e': 1000.0,
    'EX_cobalt2_e': 1000.0,
    'EX_cu2_e': 1000.0,
    'EX_fe2_e': 1000.0,
    'EX_fe3_e': 1000.0,
    'EX_h_e': 1000.0,
    'EX_h2o_e': 1000.0,
    'EX_k_e': 1000.0,
    'EX_mg2_e': 1000.0,
    'EX_mn2_e': 1000.0,
    'EX_n2_e': 1000.0,
    'EX_na1_e': 1000.0,
    'EX_nh4_e': 1000.0,
    'EX_o2_e': 1000.0,
    'EX_pi_e': 1000.0,
    'EX_so4_e': 1000.0,
    'EX_zn2_e': 1000.0,
    'EX_ca2_e': 1000.0,
    'EX_cl_e': 1000.0,
}


biolog_52 = pd.read_excel(BIOLOG_FILE, index_col=22)

# Keep the original Biolog ID -> BiGG ID mapping
reaction_map = {}

for reaction in biolog_52.index:
    if isinstance(reaction, str):
        reaction_bigg = reaction.replace("(e)", "_e")
        reaction_map[reaction_bigg] = reaction

sources = list(reaction_map.keys())
pret_model = cobra.io.read_sbml_model(ORIGINAL_MODEL)

valid_sources = []

for source in sources:
    try:
        pret_model.reactions.get_by_id(source)
        valid_sources.append(source)

    except KeyError:
        print(f"{source} not found in PRET model.")

df_pret = run_protein_pool_sensitivity(
    model=pret_model,
    sources=valid_sources,
    base_media=base_media,
    calibrated_pool=0.082,
)

# --------------------------------------------------------------
# ADD EXPERIMENTAL DATA
# --------------------------------------------------------------

carbon_sources = []
mean_max_rates = []

for source in valid_sources:
    reaction_original = reaction_map[source]
    carbon_sources.append(biolog_52.at[reaction_original, "carbon_source"])
    mean_max_rates.append(biolog_52.at[reaction_original, "mean_max_rate"])

df_pret.insert(0, "carbon_source", carbon_sources)
df_pret.insert(1, "mean_max_rate", mean_max_rates)
df_pret.to_csv(OUTPUT_FILE)

df_pret

EX_2hyoxplac_e not found in PRET model.
EX_3hphac_e not found in PRET model.

Running pool_0.50x: DM_usage = -0.041

Running pool_0.75x: DM_usage = -0.0615

Running pool_1.00x: DM_usage = -0.082

Running pool_1.25x: DM_usage = -0.10250000000000001

Running pool_1.50x: DM_usage = -0.123


,carbon_source,mean_max_rate,pool_0.50x,pool_0.75x,pool_1.00x,pool_1.25x,pool_1.50x
EX_succ_e,Succinic Acid,0.174076,0.041230,0.061845,0.082460,0.103075,0.123690
EX_gal_e,D-Galactose,0.106079,0.034996,0.052493,0.069991,0.087489,0.104987
EX_asp__L_e,L-Aspartic Acid,0.141468,0.039755,0.059633,0.079510,0.099388,0.119265
EX_pro__L_e,L-Proline,0.170164,0.037135,0.055702,0.074269,0.092837,0.111404
EX_ser__D_e,D-Serine,0.113648,0.036785,0.055178,0.073571,0.091963,0.110356
EX_glyc_e,Glycerol,0.155572,0.044714,0.067071,0.089427,0.111784,0.134141
EX_glcn_e,D-Gluconic Acid,0.123262,0.041288,0.061932,0.082575,0.103219,0.123863
EX_glyc3p_e,"D,L-alpha-Glycerol Phosphate",0.225251,0.055360,0.083040,0.110720,0.138399,0.166079
EX_mnl_e,D-Mannitol,0.134823,0.044360,0.066539,0.088719,0.110899,0.133079
EX_glu__L_e,L-Glutamic Acid,0.176334,0.045763,0.068645,0.091526,0.114408,0.137290
